# Cross-Regional Housing Price Prediction & Generalization Study
## Phase 2 — Preprocessing & Feature Engineering

In [127]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# Load utils
with open('/content/drive/MyDrive/Housing_Project/utils.py', 'r') as f:
    exec(f.read())

ames, kc = load_raw()
print(ames.shape, kc.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(2930, 82) (21613, 21)


In [128]:
df = ames.copy()
print(df.shape)

(2930, 82)


###2.1 Drop useless columns
* Misc Feature (96.4% missing)
* Mas Vnr Type (60.6% missing)
* Fireplace Qu (48.5% missing) → drop, already have Fireplaces column

In [129]:
df.drop(columns=['Misc Feature', 'Mas Vnr Type','Fireplace Qu'], inplace=True)
print(df.shape)

(2930, 79)


###2.2 Handle missing values

Convert to Binary (NaN = absence, not unknown):
* Pool QC (99.6% missing) → HasPool
* Fence (80.5% missing) → HasFence
* Alley (93.2% missing) → HasAlley


In [130]:
df['HasPool'] = df['Pool QC'].notnull().astype(int)
df.drop(columns=['Pool QC'], inplace=True)

In [131]:
df['HasFence'] = df['Fence'].notnull().astype(int)
df.drop(columns='Fence', inplace=True)

In [132]:
df['HasAlley'] = df['Alley'].notnull().astype(int)
df.drop(columns=['Alley'], inplace=True)

In [133]:
df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'].transform(lambda x: x.fillna(x.median()))

In [134]:
df['Mas Vnr Area'] = df['Mas Vnr Area'].fillna(0)

In [135]:
df['Garage Yr Blt'] = df['Garage Yr Blt'].fillna(df['Year Built'])

In [136]:
#Bsmt categorical → fill "None"
bsmt_cat = ['Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2']

df[bsmt_cat] = df[bsmt_cat].fillna('None')

In [137]:
#Bsmt numeric → fill 0
bsmt_num = ['BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath']

df[bsmt_num] = df[bsmt_num].fillna(0)

In [138]:
#Garage categorical → fill "None"
garage_cat = ['Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond']

df[garage_cat] = df[garage_cat].fillna('None')

In [139]:
#Garage numeric → fill 0
garage_num = ['Garage Cars', 'Garage Area']

df[garage_num] = df[garage_num].fillna(0)

In [140]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Lot Frontage    3
Electrical      1
dtype: int64


In [141]:
df['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].median())

In [142]:
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [143]:
print(df.isnull().sum()[df.isnull().sum() > 0])

Series([], dtype: int64)


### 2.3 Feature Engineering
Creating 4 new features from existing ones:
- `TotalSF` — combined area (Basement + Living + Garage)
- `HouseAge` — age of house when sold
- `IsRemodeled` — was it remodeled before sale? (1/0)
- `TotalBath` — combined bathroom count (half baths = 0.5)

In [144]:
df['TotalSF'] = df['Total Bsmt SF'] + df['Gr Liv Area'] + df['Garage Area']

df['HouseAge'] = df['Yr Sold'] - df['Year Built']

df['IsRemodeled'] = (df['Year Remod/Add'] != df['Year Built']).astype(int)

df['TotalBath'] = df['Full Bath'] + df['Bsmt Full Bath'] + 0.5 * df['Half Bath'] + 0.5 * df['Bsmt Half Bath']

### 2.4 Ordinal Encoding
Columns with natural order → mapped to numbers manually.

**Quality columns** (shared map: None=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5):
Exter Qual/Cond, Bsmt Qual/Cond, Kitchen Qual, Heating QC, Garage Qual/Cond

**Other ordinal columns** (custom maps):
- Garage Finish: Unf=1, RFn=2, Fin=3
- Bsmt Exposure: None=0, No=1, Mn=2, Av=3, Gd=4
- BsmtFin Type 1/2: None=0 → GLQ=6
- Lot Shape: IR3=1 → Reg=4
- Land Slope: Sev=1, Mod=2, Gtl=3

In [145]:
quality_cols = ['Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond',
                'Kitchen Qual', 'Heating QC', 'Garage Qual', 'Garage Cond']

quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

for col in quality_cols:
    df[col] = df[col].map(quality_map)

In [146]:
bsmt_fin_map = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
for col in ['BsmtFin Type 1', 'BsmtFin Type 2']:
    df[col] = df[col].map(bsmt_fin_map)

In [147]:
df['Garage Finish'] = df['Garage Finish'].map({'None':0,'Unf':1, 'RFn':2, 'Fin':3})
df['Bsmt Exposure'] = df['Bsmt Exposure'].map({'None':0, 'No':1, 'Mn':2, 'Av':3, 'Gd':4})
df['Lot Shape'] = df['Lot Shape'].map({'IR3':1, 'IR2':2, 'IR1':3, 'Reg':4})
df['Land Slope'] = df['Land Slope'].map({'Sev':1, 'Mod':2, 'Gtl':3})

### 2.5 One-Hot Encoding

Columns encoded: MS Zoning, Street, Land Contour, Utilities, Lot Config,
Condition 1/2, Bldg Type, House Style, Roof Style, Roof Matl, Foundation,
Heating, Central Air, Sale Type, Sale Condition, Garage Type, MS SubClass,
Electrical, Functional, Paved Drive

Note: drop='first' to avoid dummy variable trap (multicollinearity)
Shape went from 79 → 175 columns after encoding

In [148]:
from sklearn.preprocessing import OneHotEncoder

onehot_cols = [
    'MS Zoning', 'Street', 'Land Contour', 'Utilities',
    'Lot Config', 'Condition 1', 'Condition 2', 'Bldg Type',
    'House Style', 'Roof Style', 'Roof Matl', 'Foundation',
    'Heating', 'Central Air', 'Sale Type', 'Sale Condition',
    'Garage Type', 'MS SubClass','Electrical', 'Functional', 'Paved Drive'
]

ohe = OneHotEncoder(sparse_output=False, drop='first')
ohe_encoded = ohe.fit_transform(df[onehot_cols])

In [150]:
# Convert encoded array to dataframe with proper column names
ohe_df = pd.DataFrame(ohe_encoded,
                      columns=ohe.get_feature_names_out(onehot_cols),
                      index=df.index)

# Drop original columns
df = df.drop(columns=onehot_cols)

# Concatenate new columns
df = pd.concat([df, ohe_df], axis=1)

print(df.shape)

(2930, 175)


In [151]:
print(df.dtypes.value_counts())
print("\nAny remaining object columns:")
print(df.select_dtypes(include='object').columns.tolist())

float64    126
int64       46
object       3
Name: count, dtype: int64

Any remaining object columns:
['Neighborhood', 'Exterior 1st', 'Exterior 2nd']


### 2.6 Train/Test Split

Splitting BEFORE target encoding to avoid data leakage.
80% train (2344 rows), 20% test (586 rows)

In [152]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

In [153]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train: (2344, 174)
X_test: (586, 174)
y_train: (2344,)
y_test: (586,)


### 2.7 Target Encoding (High cardinality)
- Neighborhood (28 categories)
- Exterior 1st, Exterior 2nd (16-17 categories)
- NOTE: fit on train only to avoid data leakage!

Used category_encoders library (more reliable than sklearn's TargetEncoder
for standalone use).

In [154]:
!pip install category_encoders

from category_encoders import TargetEncoder

target_cols = ['Neighborhood', 'Exterior 1st', 'Exterior 2nd']

te = TargetEncoder(cols=target_cols)

X_train[target_cols] = te.fit_transform(X_train[target_cols], y_train)
X_test[target_cols] = te.transform(X_test[target_cols])

print(X_train[target_cols].head())

       Neighborhood   Exterior 1st   Exterior 2nd
381   186081.133193  160095.465565  160199.011976
834   204762.220904  168529.230346  165178.288906
1898  145624.839237  160095.465565  160199.011976
678   145624.839237  185495.859720  177880.681640
700   124756.488343  117563.262994  127115.984280


In [155]:
print(X_train.select_dtypes(include='object').columns.tolist())
print(X_test.select_dtypes(include='object').columns.tolist())

[]
[]


In [156]:
print(X_train['Neighborhood'].head())

381     186081.133193
834     204762.220904
1898    145624.839237
678     145624.839237
700     124756.488343
Name: Neighborhood, dtype: float64


### 2.8 Feature Scaling

Used StandardScaler (not MinMaxScaler) because:
- Our data has outliers (luxury houses)
- StandardScaler gives mean=0, std=1

IMPORTANT: fit on train only, transform both.
35 continuous columns scaled out of 174 total.

In [157]:
# columns that are already binary (0/1)
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2]
print(f"Binary columns: {len(binary_cols)}")

# columns with small range (ordinal 0-6)
ordinal_cols = [col for col in X_train.columns if X_train[col].nunique() <= 7 and col not in binary_cols]
print(f"Ordinal-like columns: {len(ordinal_cols)}")

# everything else = continuous, needs scaling
scale_cols = [col for col in X_train.columns if col not in binary_cols and col not in ordinal_cols]
print(f"Columns to scale: {len(scale_cols)}")

Binary columns: 117
Ordinal-like columns: 22
Columns to scale: 35


In [158]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

### 2.9 Final Check + Save

Sanity check before saving:
- Zero missing values
- 174 features, all numeric
- No object columns remaining
- Train: 2344 rows | Test: 586 rows

Saving processed data to Drive for use in future notebooks.

In [159]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nAny missing values in X_train: {X_train.isnull().sum().sum()}")
print(f"Any missing values in X_test: {X_test.isnull().sum().sum()}")
print(f"\nAny object columns remaining: {X_train.select_dtypes(include='object').columns.tolist()}")

X_train shape: (2344, 174)
X_test shape: (586, 174)

Any missing values in X_train: 0
Any missing values in X_test: 0

Any object columns remaining: []


In [163]:
save_processed(X_train, 'X_train.csv')
save_processed(X_test, 'X_test.csv')
y_train.to_csv(os.path.join(PROCESSED, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(PROCESSED, 'y_test.csv'), index=False)

print("files saved!")

Saved: X_train.csv
Saved: X_test.csv
files saved!
